# A Practical Primer on Python `async` / `await`

You just saw `3.multi-agent-eventplanner.ipynb` run its sub-agents **one after another**, and
`4.multi-agent-eventplanner-parallel.ipynb` run them **at the same time** with `asyncio.gather`.
This notebook explains the machinery behind that speed-up so the parallel notebook makes total sense.

**What you'll learn**
1. Why async helps (I/O-bound vs CPU-bound work)
2. Coroutines: `async def` and `await`
3. Running things concurrently: `asyncio.gather`, `create_task`, `as_completed`
4. Handling errors and timeouts
5. Limiting concurrency with a `Semaphore` (important for API rate limits)
6. Not blocking the event loop: `asyncio.to_thread`
7. Tying it all back to LangChain's `invoke` vs `ainvoke`

Every example is self-contained and uses `asyncio.sleep()` to *simulate* a slow network call,
so nothing here needs the model or any API keys. Run the cells top to bottom.

## 0. The one idea to hold on to

> **`async` does not make your code run on more CPUs. It lets a single thread do something useful
> while it would otherwise be _waiting_.**

There are two kinds of slow work:

| Kind | Example | Does async help? |
|------|---------|------------------|
| **I/O-bound** (waiting) | calling an LLM endpoint, a web search, a DB query | **Yes** &mdash; overlap the waits |
| **CPU-bound** (computing) | training a model, crunching numbers | No &mdash; use multiprocessing |

An agent step is almost entirely *waiting on the network*, so it is the perfect case for async.

## 1. The problem: synchronous code waits, then waits again

`time.sleep(2)` below stands in for a 2-second network call. Two of them run back-to-back,
so the total is ~4 seconds. The program is doing nothing but waiting for most of that time.

In [ ]:
import time

def slow_call_sync(name, seconds):
    print(f"  start  {name}")
    time.sleep(seconds)          # blocks the whole program
    print(f"  finish {name}")
    return f"{name} result"

t0 = time.perf_counter()
a = slow_call_sync("venue", 2)
b = slow_call_sync("playlist", 2)
print(f"Total: {time.perf_counter() - t0:.1f}s")   # ~4.0s

## 2. Coroutines: `async def` and `await`

- A function defined with **`async def`** is a **coroutine function**. Calling it does *not* run it &mdash;
  it returns a **coroutine object** (a paused computation).
- **`await`** runs a coroutine to completion and gives you its result. Crucially, while an `await`
  is waiting on I/O, the event loop is free to run *other* coroutines.
- `asyncio.sleep(n)` is the async version of `time.sleep(n)`: it pauses **this** coroutine without
  blocking the whole thread.

In [ ]:
import asyncio

async def slow_call(name, seconds):
    print(f"  start  {name}")
    await asyncio.sleep(seconds)   # pause THIS coroutine, don't block the thread
    print(f"  finish {name}")
    return f"{name} result"

# Calling it just creates a coroutine object - notice nothing runs yet:
coro = slow_call("venue", 2)
print(type(coro))

# 'await' actually runs it. Top-level await works directly in a Jupyter cell.
result = await coro
print(result)

### `await` vs `asyncio.run`

- **Inside Jupyter** a notebook already runs inside an event loop, so you can write `await ...`
  at the top level of a cell (as above).
- **In a plain `.py` script** there is no running loop, so you start one with **`asyncio.run(main())`**,
  where `main` is an `async def`. You typically have exactly one `asyncio.run(...)` at the entry point.

```python
# script.py
import asyncio
async def main():
    print(await slow_call("venue", 1))
asyncio.run(main())   # <- do NOT call this inside Jupyter; use bare 'await' instead
```

## 3. `await`ing sequentially still does not overlap

Awaiting one coroutine and *then* the next is still sequential &mdash; you told Python to finish the
first before starting the second. Same ~4s as the sync version.

In [ ]:
t0 = time.perf_counter()
a = await slow_call("venue", 2)
b = await slow_call("playlist", 2)
print(f"Total: {time.perf_counter() - t0:.1f}s")   # ~4.0s - no speed-up

## 4. `asyncio.gather` &mdash; run coroutines concurrently  &#11088;

`gather` schedules **all** the coroutines you give it and waits for **all** to finish, letting their
waits overlap. Results come back **in the same order** you passed them, regardless of who finished first.

This is the exact call used in notebook 4 to run the venue and playlist sub-agents together.

In [ ]:
t0 = time.perf_counter()
a, b = await asyncio.gather(
    slow_call("venue", 2),
    slow_call("playlist", 2),
)
print(f"Total: {time.perf_counter() - t0:.1f}s   ->  {a} | {b}")   # ~2.0s

Notice both `start` lines print immediately, the two sleeps overlap, and the total collapses to the
**slower** of the two (`max`) instead of the **sum**. With N independent tasks the win grows with N.

You can also gather a **dynamic list** of tasks with argument unpacking:

In [ ]:
jobs = [slow_call(f"task-{i}", 1) for i in range(5)]
t0 = time.perf_counter()
results = await asyncio.gather(*jobs)   # note the * to unpack the list
print(results)
print(f"5 one-second tasks finished in {time.perf_counter() - t0:.1f}s")   # ~1.0s"

## 5. `create_task` &mdash; start now, await later

`asyncio.create_task(coro)` schedules a coroutine to start running **immediately** in the background
and hands you a `Task` handle. You do other work, then `await` the handle when you need the result.
`gather` uses this under the hood; use `create_task` directly when you want to kick something off
early and collect it later.

In [ ]:
t0 = time.perf_counter()
venue_task    = asyncio.create_task(slow_call("venue", 2))     # starts now
playlist_task = asyncio.create_task(slow_call("playlist", 2))  # starts now

print("both tasks are already running while we do this print...")

venue    = await venue_task      # collect results
playlist = await playlist_task
print(f"Total: {time.perf_counter() - t0:.1f}s")   # ~2.0s

## 6. `as_completed` &mdash; react as each result lands

When some tasks finish sooner than others and you want to process each result the moment it is ready
(instead of waiting for the whole batch), iterate with `asyncio.as_completed`.

In [ ]:
tasks = [
    slow_call("fast",  1),
    slow_call("slow",  3),
    slow_call("medium", 2),
]
t0 = time.perf_counter()
for coro in asyncio.as_completed(tasks):
    result = await coro   # yields in COMPLETION order: fast, medium, slow
    print(f"  got '{result}' at {time.perf_counter() - t0:.1f}s")

## 7. Errors: `gather(..., return_exceptions=True)`

By default, if any gathered coroutine raises, `gather` propagates that exception immediately.
Passing `return_exceptions=True` instead **returns** each exception in the results list, so one
failing sub-agent doesn't sink the whole batch &mdash; you can inspect and handle failures per task.

In [ ]:
async def flaky(name, seconds, fail=False):
    await asyncio.sleep(seconds)
    if fail:
        raise ValueError(f"{name} blew up")
    return f"{name} ok"

results = await asyncio.gather(
    flaky("venue", 1),
    flaky("playlist", 1, fail=True),
    flaky("catering", 1),
    return_exceptions=True,
)
for r in results:
    print("  ERROR:", r) if isinstance(r, Exception) else print("  OK:   ", r)

## 8. Timeouts: `asyncio.wait_for`

Never let a hung network call block forever. `asyncio.wait_for(coro, timeout=...)` cancels the
coroutine and raises `TimeoutError` if it runs too long.

In [ ]:
try:
    await asyncio.wait_for(slow_call("stuck", 5), timeout=2)
except asyncio.TimeoutError:
    print("Timed out after 2s - the coroutine was cancelled.")

## 9. Limiting concurrency: `asyncio.Semaphore`  (rate limits!)

`gather` will happily launch **all** tasks at once. Real model endpoints have **rate limits**, so
firing 100 sub-agents simultaneously can get you throttled. A `Semaphore(k)` caps how many run at
the same time &mdash; the rest wait their turn. This is the practical way to say *'parallel, but at
most k in flight'*.

In [ ]:
sem = asyncio.Semaphore(2)   # at most 2 running concurrently

async def limited_call(name, seconds):
    async with sem:                     # acquire a slot (waits if full)
        return await slow_call(name, seconds)

t0 = time.perf_counter()
await asyncio.gather(*[limited_call(f"job-{i}", 1) for i in range(6)])
print(f"6 one-second jobs, 2 at a time -> {time.perf_counter() - t0:.1f}s")   # ~3.0s (3 waves)

## 10. Do not block the event loop: `asyncio.to_thread`

The event loop is single-threaded. If you call a **blocking** function (`time.sleep`, heavy CPU,
a synchronous library) inside a coroutine, you freeze *everything* &mdash; no overlap happens.

- Prefer the async version of a library when it exists (e.g. `ainvoke` instead of `invoke`).
- If you must call blocking code, offload it to a worker thread with **`asyncio.to_thread(fn, *args)`**
  so the loop stays free.

In [ ]:
def blocking_cpu_ish(name, seconds):
    time.sleep(seconds)     # a blocking, synchronous function
    return f"{name} done"

t0 = time.perf_counter()
# Offload both blocking calls to threads so they overlap:
a, b = await asyncio.gather(
    asyncio.to_thread(blocking_cpu_ish, "venue", 2),
    asyncio.to_thread(blocking_cpu_ish, "playlist", 2),
)
print(f"{a} | {b}  in {time.perf_counter() - t0:.1f}s")   # ~2.0s

## 11. Bringing it back to LangChain agents

Everything above maps directly onto notebook 4. LangChain runnables (models, agents built with
`create_agent`) expose **both** a synchronous and an asynchronous API:

| Synchronous (blocks) | Asynchronous (awaitable) |
|----------------------|--------------------------|
| `agent.invoke(...)`  | `await agent.ainvoke(...)` |
| `agent.stream(...)`  | `async for ... in agent.astream(...)` |
| `agent.batch([...])` | `await agent.abatch([...])` |

So the sequential planner does the equivalent of the left column, one after another:

```python
v = venue_agent.invoke({"messages": [...]})      # wait ~T1
p = playlist_agent.invoke({"messages": [...]})   # then wait ~T2   -> T1 + T2
```

and the parallel planner overlaps the two waits with exactly the `gather` you learned here:

```python
v, p = await asyncio.gather(
    venue_agent.ainvoke({"messages": [...]}),
    playlist_agent.ainvoke({"messages": [...]}),
)                                                 # -> max(T1, T2)
```

The cell below simulates that with `slow_call` so you can see the identical shape without the model.

In [ ]:
# Stand-ins for venue_agent.ainvoke(...) and playlist_agent.ainvoke(...)
async def venue_agent_ainvoke():    return await slow_call("venue-agent", 3)
async def playlist_agent_ainvoke(): return await slow_call("playlist-agent", 2)

# Sequential (what notebook 3 does)
t0 = time.perf_counter()
v = await venue_agent_ainvoke()
p = await playlist_agent_ainvoke()
print(f"Sequential: {time.perf_counter() - t0:.1f}s")   # ~5.0s

# Parallel (what notebook 4 does)
t0 = time.perf_counter()
v, p = await asyncio.gather(venue_agent_ainvoke(), playlist_agent_ainvoke())
print(f"Parallel:   {time.perf_counter() - t0:.1f}s")   # ~3.0s (the slower one)

## 12. Cheat-sheet & gotchas

**Core vocabulary**
- `async def` -> defines a **coroutine function**; calling it returns a coroutine (nothing runs yet).
- `await x` -> run coroutine/awaitable `x`, yield control while it waits, then return its result.
- `asyncio.gather(*coros)` -> run many concurrently, wait for all, results **in input order**.
- `asyncio.create_task(coro)` -> start a coroutine **now** in the background; `await` the handle later.
- `asyncio.as_completed(coros)` -> iterate results **as they finish**.
- `asyncio.wait_for(coro, timeout)` -> cancel + raise `TimeoutError` if too slow.
- `asyncio.Semaphore(k)` -> cap concurrency to `k` at once (respect rate limits).
- `asyncio.to_thread(fn, ...)` -> run a **blocking** function without freezing the loop.
- `asyncio.run(main())` -> start the loop from a **script** (not needed inside Jupyter).

**Gotchas**
- Forgetting `await` -> you get a coroutine object (and a `RuntimeWarning`), not a result.
- Awaiting in sequence -> no overlap; you must `gather`/`create_task` for concurrency.
- Calling blocking code (`time.sleep`, sync SDKs) in a coroutine -> freezes everything; use the async
  API or `to_thread`.
- Async only helps **I/O-bound** work. For CPU-bound work use processes, not async.
- Concurrency is not free: you pay for all in-flight calls at once and can hit rate limits &mdash; cap it
  with a `Semaphore`.

Now revisit `4.multi-agent-eventplanner-parallel.ipynb` &mdash; every line there should read naturally.